In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/finclub-open-project-26/sandbox_solution.csv
/kaggle/input/competitions/finclub-open-project-26/submission-converter.ipynb
/kaggle/input/competitions/finclub-open-project-26/dataset.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.decomposition import PCA

In [3]:
df = pd.read_csv("/kaggle/input/competitions/finclub-open-project-26/dataset.csv")

option_cols = [
    c for c in df.columns
    if c not in ["datetime", "underlying_price"]
]

missing_mask = df[option_cols].isna()

In [4]:
filled = df.copy()

filled[option_cols] = (
    filled[option_cols]
    .interpolate(
        axis=1,
        method="linear",
        limit_direction="both"
    )
)

filled[option_cols] = (
    filled[option_cols]
    .interpolate(
        axis=0,
        method="linear",
        limit_direction="both"
    )
)

In [5]:
def iterative_pca_completion(
    X_init,
    missing_mask,
    n_components,
    n_iter=20
):

    X = X_init.copy()

    mask = missing_mask.values

    for _ in range(n_iter):

        pca = PCA(
            n_components=n_components,
            random_state=42
        )

        X_pca = pca.fit_transform(X)

        X_reconstructed = pca.inverse_transform(
            X_pca
        )

        X[mask] = X_reconstructed[mask]

    return X

In [6]:
X_base = filled[option_cols].values

X2 = iterative_pca_completion(
    X_base,
    missing_mask,
    n_components=2,
    n_iter=20
)

In [7]:
X3 = iterative_pca_completion(
    X_base,
    missing_mask,
    n_components=3,
    n_iter=20
)

In [8]:
X4 = iterative_pca_completion(
    X_base,
    missing_mask,
    n_components=4,
    n_iter=20
)

In [9]:
X5 = iterative_pca_completion(
    X_base,
    missing_mask,
    n_components=5,
    n_iter=20
)

In [10]:
X6 = iterative_pca_completion(
    X_base,
    missing_mask,
    n_components=6,
    n_iter=20
)

In [11]:
X7 = iterative_pca_completion(
    X_base,
    missing_mask,
    n_components=7,
    n_iter=20
)

In [12]:
X8 = iterative_pca_completion(
    X_base,
    missing_mask,
    n_components=8,
    n_iter=20
)

In [13]:
X_ensemble = (
    0.10 * X2 +
    0.45 * X3 +
    0.20 * X4 +
    0.10 * X5 +
    0.05 * X6 +
    0.05 * X7 +
    0.05 * X8
)

In [14]:
final_df = df.copy()

for col_idx, col in enumerate(option_cols):

    mask_col = missing_mask[col]

    final_df.loc[
        mask_col,
        col
    ] = X_ensemble[
        mask_col.values,
        col_idx
    ]

In [15]:
final_df.to_csv(
    "filled_dataset.csv",
    index=False
)

print("saved")

saved


In [16]:


ORIGINAL_DATASET_PATH = "/kaggle/input/competitions/finclub-open-project-26/dataset.csv"

SEPARATOR = "||"

In [17]:
def generate_solution(
    filled_path: str,
    output_path: str = "submission11.csv"
):

    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled = pd.read_csv(filled_path)

    feature_cols = [
        c for c in original.columns
        if c != "datetime"
    ]

    rows = []

    for col in feature_cols:

        was_missing = original[col].isna()

        for idx in original.index[was_missing]:

            dt = original.loc[idx, "datetime"]

            uid = f"{dt}{SEPARATOR}{col}"

            val = filled.loc[idx, col]

            rows.append({
                "id": uid,
                "value": val
            })

    solution = pd.DataFrame(
        rows,
        columns=["id", "value"]
    )

    solution = (
        solution
        .sort_values("id")
        .reset_index(drop=True)
    )

    solution.to_csv(
        output_path,
        index=False
    )

    print(
        f"✅ Solution saved → {output_path} ({len(solution)} rows)"
    )

In [18]:
generate_solution(
    "filled_dataset.csv",
    "submission11.csv"
)

✅ Solution saved → submission11.csv (5460 rows)
